<a href="https://colab.research.google.com/github/ruicatzzz/aigc-detector/blob/daphne/notebooks/colab_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%cd /Users/daphnechia/Documents/GitHub/aigc-detector

# change to ur own file path

In [ ]:
import kagglehub
path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print(path)

In [ ]:
import shutil, os

# path is whatever printed from the previous cell
os.makedirs('data/cifake', exist_ok=True)
shutil.copytree(path, 'data/cifake', dirs_exist_ok=True)

In [ ]:
#train on cifake

!python -m src.train --data_dir data/cifake/train --epochs 5 --out checkpoints/cnn_cifake.pt

In [ ]:
#checking current accuracy

!python -m src.infer --input_dir data/cifake/test --output_json outputs/preds.json --checkpoint checkpoints/cnn_cifake.pt

In [ ]:
#checking the first 20 predictions

!cat outputs/preds.json | head -20

In [ ]:
#load partial dataset from huggingface

from datasets import load_dataset
from pathlib import Path
import os

N_SAMPLES = 10000  # adjust based on how much you want

out_dir = Path("data/sid_subset")
(out_dir / "REAL").mkdir(parents=True, exist_ok=True)
(out_dir / "FAKE").mkdir(parents=True, exist_ok=True)

ds = load_dataset("saberzl/SID_Set", split="train", streaming=True)

real_count, fake_count = 0, 0
for i, example in enumerate(ds):
    if i >= N_SAMPLES:
        break
    label = example["label"]  # 0=real, 1=full_synthetic, 2=tampered
    img = example["image"]    # PIL Image directly

    if label == 0:
        img.convert("RGB").save(out_dir / "REAL" / f"{example['img_id']}.jpg")
        real_count += 1
    else:  # both full_synthetic and tampered count as AIGC/FAKE for this task
        img.convert("RGB").save(out_dir / "FAKE" / f"{example['img_id']}.jpg")
        fake_count += 1

print(f"Saved {real_count} REAL, {fake_count} FAKE images to {out_dir}")

In [ ]:
#train on sid_subset

!python -m src.train --data_dir data/sid_subset --epochs 10 --out checkpoints/cnn_sid.pt

In [ ]:
#check how well the model prediction is currently

!python -m src.robustness_test